# Baseline ControlNet Depth Pipeline

This notebook runs the baseline thermal-to-RGB reconstruction pipeline without applying TADN. It uses the raw thermal-derived depth maps as ControlNet depth conditioning inputs and generates RGB outputs using Stable Diffusion 1.5. The notebook establishes the control condition used for comparison against the TADN-enhanced pipeline.

In [3]:
%pip install -q diffusers transformers accelerate safetensors


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import gc
import torch
from PIL import Image
from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline, DPMSolverMultistepScheduler

cwd = Path.cwd()
ROOT = None
for p in [cwd] + list(cwd.parents):
    if (p / "Thermal_Depth_Sandbox").exists():
        ROOT = p / "Thermal_Depth_Sandbox"
        break
if ROOT is None:
    raise FileNotFoundError("Could not locate Thermal_Depth_Sandbox from current working directory.")

THERMAL_DIR = ROOT / "input_thermal"
RAW_DEPTH_DIR = ROOT / "depth_outputs"
OUT_DIR = ROOT / "outputs_baseline"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("THERMAL_DIR:", THERMAL_DIR)
print("RAW_DEPTH_DIR:", RAW_DEPTH_DIR)
print("OUT_DIR:", OUT_DIR)


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROOT: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox
THERMAL_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal
RAW_DEPTH_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs
OUT_DIR: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline


/opt/homebrew/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/homebrew/lib/python3.11/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/opt/homebrew/lib/python3.11/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)


In [ ]:
TARGET_SIZE = (384, 384)
STEPS = 14
GUIDANCE = 5.5
STRENGTH = 0.70
CONTROL_SCALE = 0.80
SEED = 123

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32 if device in ["mps", "cpu"] else torch.float16

print("Device:", device, "| dtype:", dtype)


Device: mps | dtype: torch.float32


In [6]:
PROMPTS = {
    "t1": {
        "basic": "A photorealistic daytime suburban road scene from a dashcam, with natural colors, blue sky, green trees, gray asphalt road, utility poles, parked vehicles, and realistic daylight.",
        "structured": "A photorealistic daytime suburban road from a dashcam perspective, with trees on both sides, utility poles, a white pickup truck parked on the right, distant cars ahead, and strong sun glare.",
        "thermal_aware": "photorealistic daytime suburban road, dashcam viewpoint looking straight down the street, trees lining both sides, utility poles and overhead power lines, a white pickup truck parked on the right shoulder, distant cars ahead, asphalt road with faint lane markings and crosswalk markings, strong sun glare from the horizon, slightly hazy washed-out highlights, natural realistic colors, DSLR photo, ultra realistic, same scene, same camera angle"
    },
    "t2": {
        "basic": "A daytime suburban road from a dashcam, with trees, cars, and a bright sky.",
        "structured": "A photorealistic suburban road seen from a dashcam in early morning daylight, with trees on both sides, distant cars, utility poles, road markings, and a bright overexposed sky.",
        "thermal_aware": "photorealistic full-color RGB daytime suburban road, early morning sunlight, very bright white sky with strong glare, slight haze, subtle lens flare, grey asphalt road, visible road texture, white lane markings, trees lining both left and right sides of the road, green foliage, utility poles and power lines along the right side, a few distant cars ahead on the road, realistic camera photograph, neutral daylight white balance, natural colors blue sky tint green trees grey road, not stylized"
    },
    "t3": {
        "basic": "A suburban street in daylight with cars, trees, and houses.",
        "structured": "A photorealistic early morning suburban street from a dashcam, with parked and moving cars, trees on both sides, residential houses, utility poles, power lines, and a public bench on the right sidewalk.",
        "thermal_aware": "Photorealistic early morning suburban street scene with bright white sky and strong sun glare. Grey asphalt road with white lane markings, slightly overexposed surface. Cars driving and parked along the left side, one vehicle approaching from the distance. A public bench on the right sidewalk near residential houses. Tall green trees on both sides, utility poles and overhead power lines visible. Natural daylight, real camera photograph, unedited."
    },
    "t4": {
        "basic": "A daytime suburban street from a dashcam with a car ahead, trees, and houses.",
        "structured": "A photorealistic suburban road from a dashcam in soft daylight, with one car ahead, trees lining the road, residential houses, utility poles, and mountains in the distance.",
        "thermal_aware": "photorealistic early morning suburban street scene viewed from a dashcam, soft natural daylight with pale blue sky, slightly hazy atmosphere, gray asphalt road with visible cracks and texture, a single car driving ahead in the center of the road, low residential walls and houses on both sides, tall trees lining the street, mountains visible in the distance, utility poles and overhead power lines, calm quiet neighborhood, natural muted colors, realistic unedited camera photograph"
    },
    "t5": {
        "basic": "A suburban intersection in daylight with a white SUV, another car, and trees.",
        "structured": "A realistic daytime suburban intersection from a dashcam, with a white SUV stopped near the camera, another car entering from the left, visible stop road markings, trees, houses, and utility poles.",
        "thermal_aware": "plain daytime suburban intersection, neutral lighting, extremely overexposed sky with strong sun glare, heavy haze, washed out colors, ordinary residential road at a stop, asphalt street with large white STOP marking, rear of a white SUV stopped close to the camera on the right side, another car entering from the left side of the intersection, trees and houses in the background, utility poles and overhead power lines, traffic markings visible, realistic dashcam photo, unedited, no enhancement"
    },
    "t6": {
        "basic": "A daytime suburban road with one distant car, trees, houses, and power lines.",
        "structured": "A realistic suburban street from a dashcam in daylight, with one small car in the distance, trees and hedges on both sides, houses, utility poles, power lines, and mountains in the background.",
        "thermal_aware": "plain daytime suburban street, neutral lighting, slightly overexposed sky, light haze, washed out colors, wide residential road viewed from a distant dashcam perspective, gray asphalt street with faint lane markings, a single car driving far ahead near the center of the road, trees and hedges lining both sides, detached houses set back from the street, utility poles and overhead power lines, palm trees visible along the roadside, mountains faintly visible in the background, realistic camera photo, unedited, no enhancement"
    },
    "t7": {
        "basic": "A straight suburban road in daylight with a distant car, trees, and houses.",
        "structured": "A realistic straight suburban road from a dashcam, with a single small car far ahead, trees and hedges on both sides, palm trees, houses, utility poles, and mountains in the distance.",
        "thermal_aware": "plain daytime suburban road viewed from a distant dashcam perspective, wide straight asphalt street extending far into the distance, faint center lane markings visible, a single small car driving far ahead near the center of the road, no nearby vehicles in the foreground, trees and hedges lining both sides of the street, palm trees spaced along the roadside, residential houses set back from the road, utility poles and overhead power lines, mountains visible far in the background under a pale blue sky, slightly overexposed daylight, light haze, washed-out colors, realistic unedited dashcam photograph"
    },
    "t8": {
        "basic": "A quiet suburban road in daylight with trees, a house, and mountains.",
        "structured": "A realistic daytime suburban road from a dashcam, with a wide asphalt street, large trees on the left, a suburban house on the right, a driveway, and mountains in the far background.",
        "thermal_aware": "realistic daytime dashcam photo of a quiet suburban street, wide gray asphalt road dominating the foreground, slightly curving left, soft overexposed sunlight creating haze and washed out colors, large leafy trees and tall evergreens lining the left side of the street, a low mountain ridge visible in the far background under a pale blue sky, on the right side a single-story suburban home set back from the road, visible driveway entrance and small front yard with shrubs and stone edging, subtle shadows on the pavement, natural unedited camera image"
    },
    "t9": {
        "basic": "A straight suburban road with trees, one parked car, and a bench.",
        "structured": "A realistic daytime suburban road from a dashcam, with evenly spaced trees, one parked sedan on the right, a public bench on the sidewalk, and an otherwise empty road.",
        "thermal_aware": "plain daytime suburban street, neutral daylight, long straight asphalt road with visible texture and center lane marking, trees evenly spaced on both sides forming a corridor, a single ordinary sedan parked clearly on the right side near the curb, a visible public bench placed on the right sidewalk under the trees, no people, no traffic, empty road except one parked car, slightly soft lighting, natural colors, realistic dashcam photo, unedited, no enhancement, no stylization"
    }
}

In [7]:
NEGATIVE_PROMPT = (
    "grayscale, monochrome, black and white, desaturated, low saturation, "
    "thermal, infrared, night, nighttime, dark, moody lighting, "
    "cartoon, anime, illustration, painting, CGI, render, stylized, "
    "text, watermark, logo"
)

In [8]:
def cleanup():
    gc.collect()
    if torch.backends.mps.is_available():
        try:
            torch.mps.empty_cache()
        except Exception:
            pass

def build_pipe():
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-depth",
        torch_dtype=dtype
    )
    pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(device)
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()
    pipe.vae.to(dtype=torch.float32)
    if hasattr(pipe, "unet"):
        pipe.unet.to(dtype=torch.float32 if device != "cuda" else dtype)
    if hasattr(pipe, "controlnet"):
        pipe.controlnet.to(dtype=torch.float32 if device != "cuda" else dtype)
    return pipe

cleanup()
pipe = build_pipe()
print("Baseline pipeline ready.")


Loading pipeline components...: 100%|██████████| 6/6 [00:00<00:00, 15.99it/s]


Baseline pipeline ready.


/opt/homebrew/lib/python3.11/site-packages/diffusers/pipelines/pipeline_utils.py:2186: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


In [9]:
def load_inputs(image_id: int):
    thermal_path = THERMAL_DIR / f"t{image_id}.jpeg"
    raw_depth_path = RAW_DEPTH_DIR / f"t{image_id}_depth.png"

    assert thermal_path.exists(), f"Missing thermal image: {thermal_path}"
    assert raw_depth_path.exists(), f"Missing raw depth image: {raw_depth_path}"

    thermal = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE)
    raw_depth = Image.open(raw_depth_path).convert("L").resize(TARGET_SIZE).convert("RGB")
    return thermal, raw_depth, thermal_path, raw_depth_path


In [10]:
def run_baseline(image_id: int, prompt_type: str, seed: int = SEED):
    image_key = f"t{image_id}"

    if image_key not in PROMPTS:
        raise KeyError(f"{image_key} not found in PROMPTS")

    if prompt_type not in PROMPTS[image_key]:
        raise KeyError(
            f"{prompt_type} not found for {image_key}. "
            f"Available: {list(PROMPTS[image_key].keys())}"
        )

    prompt = PROMPTS[image_key][prompt_type]

    thermal, raw_depth, thermal_path, raw_depth_path = load_inputs(image_id)

    generator = torch.Generator(device="cpu").manual_seed(seed)

    out = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=thermal,
        control_image=raw_depth,
        strength=0.28,
        num_inference_steps=30,
        guidance_scale=4.5,
        controlnet_conditioning_scale=0.80,
        generator=generator
    ).images[0]

    out_path = OUT_DIR / f"{image_key}_{prompt_type}_baseline.png"
    out.save(out_path)

    return out, out_path, thermal_path, raw_depth_path

In [11]:
from pathlib import Path

OUT_DIR = ROOT / "outputs_baseline"
OUT_DIR.mkdir(exist_ok=True)
print("Saving baseline outputs to:", OUT_DIR)

Saving baseline outputs to: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline


In [ ]:
image_id = "t1"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t1.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t1_depth.png


  5%|▍         | 1/22 [00:36<12:41, 36.26s/it]


KeyboardInterrupt: 

In [ ]:
image_id = "t1"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t1.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t1_depth.png


100%|██████████| 22/22 [29:10<00:00, 79.59s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t1_structured_baseline.png


In [ ]:
image_id = "t1"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t1.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t1_depth.png


Token indices sequence length is longer than the specified maximum sequence length for this model (84 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: [', same scene, same camera angle']
100%|██████████| 22/22 [32:06<00:00, 87.59s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t1_thermal_aware_baseline.png


In [ ]:
image_id = "t2"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t2.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t2_depth.png


100%|██████████| 22/22 [18:16<00:00, 49.82s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t2_basic_baseline.png


In [ ]:
image_id = "t2"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t2.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t2_depth.png


100%|██████████| 22/22 [32:05<00:00, 87.52s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t2_structured_baseline.png


In [ ]:
image_id = "t2"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Token indices sequence length is longer than the specified maximum sequence length for this model (98 > 77). Running this sequence through the model will result in indexing errors


Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t2.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t2_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['camera photograph, neutral daylight white balance, natural colors blue sky tint green trees grey road, not stylized']
100%|██████████| 22/22 [16:07<00:00, 43.98s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t2_thermal_aware_baseline.png


In [ ]:
image_id = "t3"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t3.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t3_depth.png


100%|██████████| 22/22 [18:24<00:00, 50.22s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t3_basic_baseline.png


In [ ]:
image_id = "t3"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t3.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t3_depth.png


100%|██████████| 22/22 [30:57<00:00, 84.45s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t3_structured_baseline.png


In [ ]:
image_id = "t3"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t3.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t3_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['real camera photograph, unedited.']
100%|██████████| 22/22 [17:20<00:00, 47.31s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t3_thermal_aware_baseline.png


In [ ]:
image_id = "t4"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t4.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t4_depth.png


100%|██████████| 22/22 [17:45<00:00, 48.45s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t4_basic_baseline.png


In [ ]:
image_id = "t4"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t4.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t4_depth.png


100%|██████████| 22/22 [18:43<00:00, 51.05s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t4_structured_baseline.png


In [ ]:
image_id = "t4"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['quiet neighborhood, natural muted colors, realistic unedited camera photograph']


Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t4.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t4_depth.png


100%|██████████| 22/22 [31:18<00:00, 85.39s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t4_thermal_aware_baseline.png


In [ ]:
image_id = "t5"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t5.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t5_depth.png


100%|██████████| 22/22 [23:39<00:00, 64.54s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t5_basic_baseline.png


In [ ]:
image_id = "t5"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t5.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t5_depth.png


100%|██████████| 22/22 [20:50<00:00, 56.82s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t5_structured_baseline.png


In [ ]:
image_id = "t5"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t5.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t5_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['and overhead power lines, traffic markings visible, realistic dashcam photo, unedited, no enhancement']
100%|██████████| 22/22 [55:45<00:00, 152.09s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t5_thermal_aware_baseline.png


In [ ]:
image_id = "t6"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t6.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t6_depth.png


100%|██████████| 22/22 [36:51<00:00, 100.53s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t6_basic_baseline.png


In [ ]:
image_id = "t6"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t6.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t6_depth.png


100%|██████████| 22/22 [36:07<00:00, 98.53s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t6_structured_baseline.png


In [ ]:
image_id = "t6"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['palm trees visible along the roadside, mountains faintly visible in the background, realistic camera photo, unedited, no enhancement']


Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t6.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t6_depth.png


100%|██████████| 22/22 [22:07<00:00, 60.32s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t6_thermal_aware_baseline.png


In [ ]:
image_id = "t7"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t7.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t7_depth.png


100%|██████████| 22/22 [28:51<00:00, 78.70s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t7_basic_baseline.png


In [ ]:
image_id = "t7"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t7.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t7_depth.png


100%|██████████| 22/22 [26:15<00:00, 71.62s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t7_structured_baseline.png


In [ ]:
image_id = "t7"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t7.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t7_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['utility poles and overhead power lines, mountains visible far in the background under a pale blue sky, slightly overexposed daylight, light haze, washed - out colors, realistic unedited dashcam photograph']
100%|██████████| 22/22 [38:59<00:00, 106.35s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t7_thermal_aware_baseline.png


In [ ]:
image_id = "t8"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t8.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t8_depth.png


100%|██████████| 22/22 [26:10<00:00, 71.39s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t8_basic_baseline.png


In [ ]:
image_id = "t8"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t8.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t8_depth.png


100%|██████████| 22/22 [33:01<00:00, 90.09s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t8_structured_baseline.png


In [ ]:
image_id = "t8"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t8.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t8_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['home set back from the road, visible driveway entrance and small front yard with shrubs and stone edging, subtle shadows on the pavement, natural unedited camera image']
100%|██████████| 22/22 [31:16<00:00, 85.32s/it] 


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t8_thermal_aware_baseline.png


In [ ]:
image_id = "t9"
prompt_type = "basic"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t9.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t9_depth.png


100%|██████████| 22/22 [22:48<00:00, 62.22s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t9_basic_baseline.png


In [ ]:
image_id = "t9"
prompt_type = "structured"   
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t9.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t9_depth.png


100%|██████████| 22/22 [1:11:37<00:00, 195.35s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t9_structured_baseline.png


In [ ]:
image_id = "t9"
prompt_type = "thermal_aware"  
seed = 123

thermal_path = THERMAL_DIR / f"{image_id}.jpeg"
raw_depth_path = RAW_DEPTH_DIR / f"{image_id}_depth.png"

print("Thermal:", thermal_path)
print("Depth:", raw_depth_path)

init_rgb = Image.open(thermal_path).convert("RGB").resize(TARGET_SIZE )
control_raw = Image.open(raw_depth_path).convert("RGB").resize(TARGET_SIZE)

prompt = PROMPTS[image_id][prompt_type]
gen = torch.Generator(device="cpu").manual_seed(seed)

img_out = pipe(
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    image=init_rgb,
    control_image=control_raw,
    strength=0.75,
    num_inference_steps=30,
    guidance_scale=8.0,
    controlnet_conditioning_scale=0.80,
    generator=gen
).images[0]

out_path = OUT_DIR / f"{image_id}_{prompt_type}_baseline.png"
img_out.save(out_path)
print("Saved:", out_path)

Thermal: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/input_thermal/t9.jpeg
Depth: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/depth_outputs/t9_depth.png


The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['natural colors, realistic dashcam photo, unedited, no enhancement, no stylization']
100%|██████████| 22/22 [25:03<00:00, 68.32s/it]


Saved: /Users/maryamellathy/Desktop/NEW thermaltoRBG/Thermal_Depth_Sandbox/outputs_baseline/t9_thermal_aware_baseline.png
